# 51 · Saída histórica por segmento

A segmentação oficial (notebook 50) rotula quem estava ativo em 30/09/2026, a data de corte. Por isso
nenhum segmento tem saída observada: depois do corte não há dado. Para medir a saída de cada grupo, este
notebook volta três meses:

1. **Foto em 30/06/2026**: quem tinha assinatura ativa nesse dia.
2. **Mesmas variáveis**, recalculadas com o que se sabia até 30/06: treinos por semana, dias sem treino
   (log), tempo de casa (log), % dos treinos ao ar livre e engajamento (janela de 90 dias, de 02/04 a 30/06).
3. **Mesma régua**: padronização e centros da segmentação oficial (`saidas/segmentacao_oficial_padronizacao.csv`
   e `_centros.csv`). Cada assinante vai para o centro mais próximo; quem não treinou nos 90 dias vai para
   "Inativos em 90 dias (regra)". O k-means não é ajustado de novo.
4. **Saída observada**: assinatura encerrada entre 01/07 e 30/09/2026.

O que o número diz: de cada 100 assinantes que tinham esse perfil em 30/06, quantos saíram nos três meses
seguintes. Não é a saída das pessoas que estão no segmento hoje.

Nada é alterado na tabela master nem na segmentação oficial. Ordem de execução: 39 → 41 → 42 → 43 → 50 → 51.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, "../src")
from engajamento import niveis_engajamento

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30"); FOTO = pd.Timestamp("2026-06-30")
AR_LIVRE = ["corrida", "caminhada", "pedal"]
NUM = ["treinos_por_semana", "dias_sem_treinar_log", "duracao_dias_log", "pct_ar_livre", "engajamento_ord"]
ORD = {"possível churn": 0, "baixo": 1, "médio": 2, "alto": 3}
ORDEM = ["Ativos ao ar livre", "Ativos em ambiente fechado", "Recém-chegados", "Esfriando", "Inativos em 90 dias (regra)"]
REGRA = "Inativos em 90 dias (regra)"

m = pd.read_parquet(P / "tabela_master.parquet")
ativ = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data", "tipo"])
ofi = pd.read_parquet(P / "segmentos_oficiais.parquet")
pad = pd.read_csv(S / "segmentacao_oficial_padronizacao.csv").set_index("variavel").loc[NUM]
cen = pd.read_csv(S / "segmentacao_oficial_centros.csv").set_index("segmento")[NUM]
print("tabela master:", m.shape, "| treinos:", len(ativ), "| segmentos oficiais:", len(ofi))
print("treinos de", ativ.data.min().date(), "a", ativ.data.max().date())
print(cen)

def atribui(d):
    # centro mais próximo, na escala padronizada da segmentação oficial
    z = (d[NUM].astype(float) - pad.media_elegiveis) / pad.desvio_elegiveis
    cz = (cen - pad.media_elegiveis) / pad.desvio_elegiveis
    dist = ((z.to_numpy()[:, None, :] - cz.to_numpy()[None, :, :]) ** 2).sum(axis=2)
    return pd.Series(cen.index[dist.argmin(axis=1)], index=d.index)

tabela master: (63929, 119) | treinos: 1252488 | segmentos oficiais: 14753
treinos de 2026-01-01 a 2026-09-30
                            treinos_por_semana  dias_sem_treinar_log  duracao_dias_log  pct_ar_livre  engajamento_ord
segmento                                                                                                             
Recém-chegados                            1.05                  0.94              3.13         56.23             2.92
Ativos em ambiente fechado                2.58                  0.86              5.77         14.83             2.95
Esfriando                                 0.86                  3.07              5.74         62.77             1.47
Ativos ao ar livre                        2.77                  0.78              5.82         80.69             2.99


## 1. Conferência da régua em 30/09

Antes de usar a régua em outra data: aplicada aos próprios elegíveis de 30/09, ela tem de devolver os
rótulos da segmentação oficial.

In [2]:
e = m[m.segmento_regra == "elegível ao k-means"].copy()
e["pct_ar_livre"] = e[["treinos_" + t for t in AR_LIVRE]].sum(axis=1) / e.treinos_90d * 100
e["engajamento_ord"] = e.nivel_engajamento.map(ORD)
e["segmento_regua"] = atribui(e)
c = e.merge(ofi, on="user_id")
iguais = int((c.segmento_regua == c.segmento).sum())
print("elegíveis em 30/09:", len(c), "| mesmo segmento pela régua:", iguais, f"({iguais / len(c):.2%})")
assert iguais / len(c) > 0.99

elegíveis em 30/09: 13999 | mesmo segmento pela régua: 13998 (99.99%)


## 2. Quem estava ativo em 30/06/2026

Cada conta tem no máximo uma assinatura. Ativa na foto: começou até 30/06 e ainda não tinha terminado
(sem data de fim, ou fim depois de 30/06).

In [3]:
a = m[m.ja_assinou].copy()
print("contas que já assinaram:", len(a))
a = a[a.data_inicio_assinatura <= FOTO]
print("-> assinatura iniciada até 30/06/2026:", len(a))
a = a[a.ativa_no_corte | (a.data_fim > FOTO)]
print("-> ainda ativa em 30/06/2026:", len(a))
f = a[["user_id", "plano", "canal_cobranca", "canal_aquisicao", "data_inicio_assinatura", "ativa_no_corte", "data_fim", "motivo_fim", "tipo_saida"]].copy().reset_index(drop=True)
f["saiu_jul_set"] = ~f.ativa_no_corte
assert f.user_id.is_unique and (f.loc[f.saiu_jul_set, "data_fim"].between(FOTO + pd.Timedelta(days=1), CORTE)).all()
print("saíram entre 01/07 e 30/09:", int(f.saiu_jul_set.sum()), f"({f.saiu_jul_set.mean():.1%})", "| seguem ativos no corte:", int((~f.saiu_jul_set).sum()))
print(f[f.saiu_jul_set].motivo_fim.value_counts().to_string())

contas que já assinaram: 32583
-> assinatura iniciada até 30/06/2026: 29375
-> ainda ativa em 30/06/2026: 14059
saíram entre 01/07 e 30/09: 2343 (16.7%) | seguem ativos no corte: 11716
motivo_fim
renovacao_desligada    2172
falha_pagamento         159
reembolso                 8
chargeback                4


## 3. As cinco variáveis, com o que se sabia até 30/06

Mesmas definições da tabela master (notebook 39) e do notebook 50, trocando a data de referência:
janela de 90 dias de 02/04 a 30/06 (0 a 89 dias antes da foto), treinos por semana = treinos ÷ (90 ÷ 7),
tempo de casa = dias do início da assinatura até 30/06, logaritmo = log(1 + x) e engajamento pela
definição do grupo (`src/engajamento.py`).

In [4]:
t = ativ[ativ.user_id.isin(f.user_id)]
print("treinos das contas da foto:", len(t))
t = t[(FOTO - t.data).dt.days.between(0, 89)]
print("-> nos 90 dias até 30/06:", len(t), "| contas com treino:", t.user_id.nunique())
g = t.groupby("user_id").agg(treinos_90d=("data", "size"), ar_livre=("tipo", lambda s: s.isin(AR_LIVRE).sum()))
top = t.groupby(["user_id", "tipo"]).size().rename("n").reset_index().sort_values(["user_id", "n"], ascending=[True, False]).drop_duplicates("user_id").set_index("user_id").tipo
n0 = len(f)
f = f.merge(g, on="user_id", how="left").merge(top.rename("tipo_principal"), on="user_id", how="left")
assert len(f) == n0
f["treinos_90d"] = f.treinos_90d.fillna(0).astype(int)
f["tipo_principal"] = f.tipo_principal.fillna("sem treino")
f["treinos_por_semana"] = (f.treinos_90d / (90 / 7)).round(2)
f["pct_ar_livre"] = f.ar_livre / f.treinos_90d.where(f.treinos_90d > 0) * 100
f["duracao_dias"] = (FOTO - f.data_inicio_assinatura).dt.days
f["meses_de_casa"] = (f.duracao_dias / 30.4375).round(1)
f["duracao_dias_log"] = np.log1p(f.duracao_dias).round(4)

ini = f.set_index("user_id").data_inicio_assinatura
en = niveis_engajamento(ativ, f.user_id, FOTO, inicio=ini)
f = f.merge(en[["user_id", "nivel", "dias_sem_treinar"]], on="user_id", how="left")
assert len(f) == n0
f["nivel_engajamento"] = f.nivel.str[3:]
f["engajamento_ord"] = f.nivel_engajamento.map(ORD)
f["dias_sem_treinar_log"] = np.log1p(f.dias_sem_treinar.astype(float)).round(4)
print("sem treino nos 90 dias:", int((f.treinos_90d == 0).sum()), "| com treino:", int((f.treinos_90d > 0).sum()))
print("nível de engajamento em 30/06:"); print(f.nivel_engajamento.value_counts().to_string())
print("vazios nas cinco variáveis, entre quem treinou:", int(f.loc[f.treinos_90d > 0, NUM].isna().sum().sum()))

treinos das contas da foto:

 996604
-> nos 90 dias até 30/06: 362244 | contas com treino: 13306


sem treino nos 90 dias: 753 | com treino: 13306
nível de engajamento em 30/06:
nivel_engajamento
alto              9660
médio             2608
possível churn    1314
baixo              477
vazios nas cinco variáveis, entre quem treinou: 0


## 4. Segmento de cada assinante em 30/06

In [5]:
el = f.treinos_90d > 0
f["segmento"] = REGRA
f.loc[el, "segmento"] = atribui(f[el])
tam = pd.DataFrame({"em 30/06 (foto)": f.segmento.value_counts(), "em 30/09 (oficial)": ofi.segmento.value_counts()}).loc[ORDEM]
tam["% em 30/06"] = tam["em 30/06 (foto)"] / len(f) * 100
tam["% em 30/09"] = tam["em 30/09 (oficial)"] / len(ofi) * 100
print(tam)
print("total:", len(f), "na foto |", len(ofi), "no corte")

                             em 30/06 (foto)  em 30/09 (oficial)  % em 30/06  % em 30/09
segmento                                                                                
Ativos ao ar livre                      4787                5001       34.05       33.90
Ativos em ambiente fechado              3659                4030       26.03       27.32
Recém-chegados                          1681                2092       11.96       14.18
Esfriando                               3179                2876       22.61       19.49
Inativos em 90 dias (regra)              753                 754        5.36        5.11
total: 14059 na foto | 14753 no corte


## 5. A tabela: saída de julho a setembro por segmento

Três fatores: **ativos** (tamanho do grupo em 30/06), **engajamento** (% no nível alto em 30/06) e
**saída** (quem encerrou a assinatura de 01/07 a 30/09). A saída aparece também por tipo e por plano,
porque o plano anual só pode sair na renovação.

In [6]:
def linha(d):
    s = d.saiu_jul_set
    me, an = d[d.plano == "mensal"], d[d.plano == "anual"]
    return pd.Series({
        "assinantes_30jun": len(d),
        "pct_dos_ativos": len(d) / len(f) * 100,
        "sairam_jul_set": int(s.sum()),
        "pct_saida": s.mean() * 100,
        "pct_saida_voluntaria": (d.tipo_saida == "voluntária").mean() * 100,
        "pct_saida_involuntaria": (d.tipo_saida == "involuntária").mean() * 100,
        "pct_das_saidas": s.sum() / f.saiu_jul_set.sum() * 100,
        "assinantes_mensal": len(me),
        "pct_saida_mensal": me.saiu_jul_set.mean() * 100,
        "assinantes_anual": len(an),
        "pct_saida_anual": an.saiu_jul_set.mean() * 100,
        "pct_engaj_alto": (d.nivel_engajamento == "alto").mean() * 100,
        "treinos_por_semana": d.treinos_por_semana.median(),
        "dias_sem_treinar": d.dias_sem_treinar.median(),
        "meses_de_casa": d.meses_de_casa.median(),
        "pct_ar_livre": d.pct_ar_livre.median(),
    })
tab = f.groupby("segmento").apply(linha).loc[ORDEM]
tab.loc["Total de ativos em 30/06"] = linha(f)
print(tab.round(1).T)
assert int(tab.loc[ORDEM, "sairam_jul_set"].sum()) == int(f.saiu_jul_set.sum())
media = tab.loc["Total de ativos em 30/06", "pct_saida"]
print()
for s_ in ORDEM:
    print(f"{s_}: {tab.loc[s_, 'pct_saida']:.1f}% de saída = {tab.loc[s_, 'pct_saida'] / media:.1f}x a média ({media:.1f}%)")

segmento                Ativos ao ar livre  Ativos em ambiente fechado  Recém-chegados  Esfriando  Inativos em 90 dias (regra)  Total de ativos em 30/06
assinantes_30jun                  4,787.00                    3,659.00        1,681.00   3,179.00                       753.00                 14,059.00
pct_dos_ativos                       34.00                       26.00           12.00      22.60                         5.40                    100.00
sairam_jul_set                      601.00                      382.00          319.00     847.00                       194.00                  2,343.00
pct_saida                            12.60                       10.40           19.00      26.60                        25.80                     16.70
pct_saida_voluntaria                 11.20                        9.50           16.40      26.10                        25.50                     15.50
pct_saida_involuntaria                1.40                        1.00            

## 6. Para onde foi cada grupo

Segmento em 30/06 (linhas) contra a situação em 30/09 (colunas): saiu, ou o segmento oficial em que está
hoje. Em % da linha.

In [7]:
d = f.merge(ofi[["user_id", "segmento"]].rename(columns={"segmento": "segmento_30set"}), on="user_id", how="left")
assert len(d) == len(f)
d["destino"] = np.where(d.saiu_jul_set, "Saiu (jul a set)", d.segmento_30set)
print("sem destino:", int(d.destino.isna().sum()))
tr = (pd.crosstab(d.segmento, d.destino, normalize="index") * 100).loc[ORDEM, ["Saiu (jul a set)"] + ORDEM]
print(tr.round(1))

sem destino: 0
destino                      Saiu (jul a set)  Ativos ao ar livre  Ativos em ambiente fechado  Recém-chegados  Esfriando  Inativos em 90 dias (regra)
segmento                                                                                                                                             
Ativos ao ar livre                      12.60               70.30                        0.20            0.10      16.00                         0.90
Ativos em ambiente fechado              10.40                0.20                       77.90            0.00      11.00                         0.50
Recém-chegados                          19.00               36.20                       24.70            2.30      16.50                         1.20
Esfriando                               26.60               18.40                       12.00            0.50      31.60                        10.90
Inativos em 90 dias (regra)             25.80                4.90                    

## 7. O gráfico: saída de julho a setembro por segmento

In [8]:
r = tab.loc[ORDEM, "pct_saida"].sort_values()
fig, ax = plt.subplots(figsize=(8, 3.6))
cores = ["#e5484d" if v > media else "#222" for v in r.values]
ax.barh([f"{i}  ({int(tab.loc[i, 'assinantes_30jun']):,} assinantes)".replace(",", ".") for i in r.index], r.values, color=cores)
ax.axvline(media, color="#e5484d", ls="--", lw=1)
ax.text(media, len(r) - 0.4, f"média dos ativos {media:.1f}%".replace(".", ","), color="#e5484d", fontsize=8)
for y, v in enumerate(r.values): ax.text(v + 0.8, y, f"{v:.1f}%".replace(".", ","), va="center", fontsize=9)
ax.set_xlabel("% dos ativos em 30/06/2026 que saíram de 01/07 a 30/09/2026"); ax.set_xlim(0, r.max() * 1.18)
ax.set_title("Saída em três meses, pelo segmento em 30/06/2026", fontsize=10, loc="left")
for s_ in ["top", "right"]: ax.spines[s_].set_visible(False)
fig.tight_layout(); fig.savefig(S / "segmentacao_oficial_saida_historica.png", dpi=130)
print("salvo: saidas/segmentacao_oficial_saida_historica.png")

salvo: saidas/segmentacao_oficial_saida_historica.png


## 8. Salvar

Limites desta medida:
- é uma janela só (julho a setembro de 2026). Os treinos começam em 01/01/2026, então a foto mais antiga
  possível com 90 dias de histórico é 31/03/2026;
- os centros foram ajustados com os ativos de 30/09 e aplicados a outra população; a divisão dos grupos
  em 30/06 não é idêntica à de hoje (seção 4);
- "Esfriando" e "Inativos" são definidos por queda de uso, que antecede a saída: a taxa alta descreve o
  grupo, não prova que a queda de uso causa a saída;
- a janela inclui a campanha de agosto de 2026, que pode ter mexido na saída de quem foi contatado.

In [9]:
tab.round(2).to_csv(S / "segmentacao_oficial_saida_historica.csv", encoding="utf-8-sig", index_label="segmento_em_30jun")
tr.round(2).to_csv(S / "segmentacao_oficial_saida_historica_destino.csv", encoding="utf-8-sig", index_label="segmento_em_30jun")
f[["user_id", "segmento", "saiu_jul_set", "tipo_saida", "plano"] + NUM].to_parquet(P / "segmentos_foto_30jun.parquet", index=False)
print("linhas gravadas:", len(f), "| tabela master e segmentos oficiais: 0 linhas alteradas")
print("arquivos: saidas/segmentacao_oficial_saida_historica.csv, _destino.csv e .png; dados/processados/segmentos_foto_30jun.parquet")

linhas gravadas: 14059 | tabela master e segmentos oficiais: 0 linhas alteradas
arquivos: saidas/segmentacao_oficial_saida_historica.csv, _destino.csv e .png; dados/processados/segmentos_foto_30jun.parquet
